In [1]:
# 1. CORE CONFIGURATION
import core_config
import os

print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
print(f"[SYSTEM] Active Conda Environment: {os.environ.get('CONDA_PREFIX')}")

[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/files
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/info
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.jupyter/lab/workspaces
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.
[SYSTEM] Active Conda Environment: /nlsasfs/home/lmedge/rahkus/Conda/envs/oec_deploy


# Phase 1: Environment Provisioning (Compute Node)

!conda create --name oec_deploy python=3.12 -y
!conda activate oec_deploy
!conda install -c conda-forge jupyter nb_conda_kernels ipykernel -y
!pip install huggingface_hub uv
!python -m ipykernel install --user --name oec_deploy --display-name "Python (oec_deploy_v12)"

In [3]:
%cd ~/oec_deploy/v1

/nlsasfs/home/lmedge/rahkus/oec_deploy/v1


!git clone https://github.com/Overworldai/Biome.git
!git clone https://github.com/Overworldai/world_engine.git

## Phase 2: Launching Jupyter (Compute Node)

1. Ensure your custom `core_config.py` file is placed directly inside the `~/oec_deploy/v1` directory so the notebook can find it.
2. Launch the Jupyter Notebook server bound to the compute node's interfaces. Note the access token printed in the terminal.
```bash
jupyter notebook --no-browser --port=8888 --ip=0.0.0.0

```



## Phase 3: The Multi-Hop Network Bridge (Ubuntu VM & Windows Host)

Because you require simultaneous access to the Jupyter environment and the Biome Inference API, you must establish two distinct tunnels.

1. **SSH Tunnels (Ubuntu VM):** Open a fresh terminal in your VirtualBox VM (outside the login node) and execute both routing commands. Replace `<COMPUTE_NODE>` with your actual Slurm node (e.g., `scn28-mn`).
```bash
# 1. Tunnel for Jupyter Interface
ssh -f -N -L 5402:<COMPUTE_NODE>:8888 alexi@login.aireos.com.in

# 2. Tunnel for Biome Server Inference API
ssh -f -N -L 7987:<COMPUTE_NODE>:7987 alexi@login.aireos.com.in

```


2. **VirtualBox Port Forwarding (Windows Host):** Ensure the following TCP rules are active in your VirtualBox Network Settings:
* **Rule 1 (Jupyter):** Host `127.0.0.1` Port `5402` $\rightarrow$ Guest `127.0.0.1` Port `5402`
* **Rule 2 (Biome API):** Host `127.0.0.1` Port `7987` $\rightarrow$ Guest `127.0.0.1` Port `7987`


## Phase 4: Jupyter Execution (`deploy_server.ipynb`)

Access Jupyter on your Windows host browser at `[http://127.0.0.1:5402](http://127.0.0.1:5402)`. Create a new notebook and select the **`Python [conda env:oec_deploy]`** kernel from the dropdown.

**Cell 1: Proxy & Core Initialization**
This must be executed first to lock in your cache mappings and proxy overrides before any ML libraries are initialized.

```python
# 1. CORE CONFIGURATION
import core_config
import os

print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
print(f"[SYSTEM] Active Conda Environment: {os.environ.get('CONDA_PREFIX')}")

```

**Cell 2: Dependency Resolution & Local Linking**
By explicitly targeting the `$CONDA_PREFIX` variable, we force the `uv` package manager to install the dependencies directly into your active Conda environment, eliminating the need for a separate `.venv`.


In [8]:
import sys
import os

python_exe = sys.executable
server_dir = os.path.expanduser("~/oec_deploy/v1/Biome/server-components")
world_engine_dir = os.path.expanduser("~/oec_deploy/v1/world_engine")

print("[SYSTEM] Purging incompatible CUDA 13.0 PyTorch...")
!uv pip uninstall --python "{python_exe}" torch torchvision torchaudio

print("\n[SYSTEM] Installing PyTorch specifically for CUDA 12.4...")
# The --index-url flag forces pip to pull from the specific PyTorch compute platform index
!uv pip install --python "{python_exe}" torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

print("\n[SYSTEM] Re-verifying Biome & World Engine dependencies...")
!uv pip install --python "{python_exe}" -e "{world_engine_dir}"
!cd "{server_dir}" && uv pip install --python "{python_exe}" .

print(f"\n[SYSTEM] Relaunching Biome inference server from: {server_dir}")
!cd "{server_dir}" && "{python_exe}" main.py --host 0.0.0.0 --port 7987

[SYSTEM] Purging incompatible CUDA 13.0 PyTorch...
Using Python 3.12.15 environment at: /nlsasfs/home/lmedge/rahkus/Conda/envs/oec_deploy
Uninstalled 2 packages in 7.28s
 - torch==2.11.0
 - torchvision==0.26.0+cu128

[SYSTEM] Installing PyTorch specifically for CUDA 12.4...
Using Python 3.12.15 environment at: /nlsasfs/home/lmedge/rahkus/Conda/envs/oec_deploy
Resolved 28 packages in 3.29s                                        
Prepared 18 packages in 39.91s                                           
Uninstalled 2 packages in 1.25s
Installed 18 packages in 5.26s                              
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4.127
 + nvidia-cuda-nvrtc-cu12==12.4.127
 + nvidia-cuda-runtime-cu12==12.4.127
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.2.1.3
 + nvidia-curand-cu12==10.3.5.147
 + nvidia-cusolver-cu12==11.6.1.9
 + nvidia-cusparse-cu12==12.3.1.170
 + nvidia-cusparselt-cu12==0.6.2
 + nvidia-nccl-cu12==2.21.5
 + nvidia-nvjitlink-cu12==12.4.1

In [9]:
import sys
import os

python_exe = sys.executable
server_dir = os.path.expanduser("~/oec_deploy/v1/Biome/server-components")

print(f"[SYSTEM] Launching Biome inference server from: {server_dir}")
print(f"[SYSTEM] Python Executable: {python_exe}")

# Explicitly use the Conda Python executable to launch the server
!cd "{server_dir}" && "{python_exe}" main.py --host 0.0.0.0 --port 7987

[SYSTEM] Launching Biome inference server from: /nlsasfs/home/lmedge/rahkus/oec_deploy/v1/Biome/server-components
[SYSTEM] Python Executable: /nlsasfs/home/lmedge/rahkus/Conda/envs/oec_deploy/bin/python
22:06:39 [info    ] [__main__] Python version=3.12.15 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:20:25) [GCC 14.3.0]
22:06:39 [info    ] [__main__] Starting server...
22:06:39 [info    ] [util.hf_token] HF token resolved and set
22:06:39 [info    ] [__main__] Basic imports done
22:06:39 [info    ] [__main__] Importing torch...
22:06:39 [error   ] [__main__] Import failed
Traceback (most recent call last):
  File "/nlsasfs/home/lmedge/rahkus/oec_deploy/v1/Biome/server-components/main.py", line 138, in <module>
    import torch
  File "/nlsasfs/home/lmedge/rahkus/Conda/envs/oec_deploy/lib/python3.12/site-packages/torch/__init__.py", line 442, in <module>
    from torch._C import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^
ImportError: /nlsasfs/home/lmedge/rahkus/Conda/envs/oec_d